# Silver Layer DDL — Table Definitions

## Overview
This notebook defines the Data Definition Language (DDL) for all **Silver Layer** tables in Databricks Delta Lake.

The Silver layer represents the cleansed, standardized, enriched, and validated data store in the Medallion Architecture. Key characteristics include:
- **Data Quality Remediations**: Primary key null filtering, deduplication, whitespace trimming, and standardization of codes/lookups.
- **Schema Evolution**: Transformation of raw fields (e.g. integer date representations converted to native `DATE`, key parsing, SCD Type 2 expiry dates).
- **Auditability**: Every table includes an audit metadata column `dwh_create_date TIMESTAMP` to track lineage and ingestion time.

### Silver Delta Lake Tables:
1. `silver.crm_cust_info` — Deduplicated and standardized CRM customer demographics with marital status and gender normalization.
2. `silver.crm_prd_info` — Normalized product catalogue with derived `cat_id`, clean `prd_key`, product line mapping, and SCD Type 2 `prd_end_dt`.
3. `silver.crm_sales_details` — Validated sales transactions with native `DATE` columns, recalculated sales amounts, and derived unit prices.
4. `silver.erp_loc_a101` — Customer geographic locations with normalized CIDs (hyphens stripped) and standardized country names.
5. `silver.erp_cust_az12` — ERP customer demographics with `'NAS'` prefix stripped, future birthdates set to null, and standardized gender.
6. `silver.erp_px_cat_g1v2` — ERP product category hierarchy and maintenance classifications with audit metadata.

All tables are stored in **Delta Lake** format (`USING DELTA`), enabling ACID compliance, time travel, and high performance.

In [ ]:
# Databricks notebook source
# COMMAND ----------
# Helper display function fallback for non-Databricks runtime environments
try:
    display
except NameError:
    def display(df, n=20):
        if hasattr(df, "show"):
            df.show(n, truncate=False)
        else:
            print(df)

# Ensure silver database exists
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

## CRM Source Tables DDL (Silver Layer)
Define Silver Delta Lake tables for CRM customers, products, and sales order lines.

In [ ]:
# COMMAND ----------
# 1. silver.crm_cust_info
spark.sql("DROP TABLE IF EXISTS silver.crm_cust_info")
spark.sql("""
CREATE TABLE silver.crm_cust_info (
    cst_id INT,
    cst_key STRING,
    cst_firstname STRING,
    cst_lastname STRING,
    cst_marital_status STRING,
    cst_gndr STRING,
    cst_create_date DATE,
    dwh_create_date TIMESTAMP
)
USING DELTA
COMMENT 'Cleaned and deduplicated CRM customer information'
""")
print(">> Table created: silver.crm_cust_info")

# 2. silver.crm_prd_info
spark.sql("DROP TABLE IF EXISTS silver.crm_prd_info")
spark.sql("""
CREATE TABLE silver.crm_prd_info (
    prd_id INT,
    cat_id STRING,
    prd_key STRING,
    prd_nm STRING,
    prd_cost INT,
    prd_line STRING,
    prd_start_dt DATE,
    prd_end_dt DATE,
    dwh_create_date TIMESTAMP
)
USING DELTA
COMMENT 'Normalized CRM product catalogue with SCD Type 2 expiry dates'
""")
print(">> Table created: silver.crm_prd_info")

# 3. silver.crm_sales_details
spark.sql("DROP TABLE IF EXISTS silver.crm_sales_details")
spark.sql("""
CREATE TABLE silver.crm_sales_details (
    sls_ord_num STRING,
    sls_prd_key STRING,
    sls_cust_id INT,
    sls_order_dt DATE,
    sls_ship_dt DATE,
    sls_due_dt DATE,
    sls_sales INT,
    sls_quantity INT,
    sls_price INT,
    dwh_create_date TIMESTAMP
)
USING DELTA
COMMENT 'Validated CRM sales order line details with native date types and integrity fixes'
""")
print(">> Table created: silver.crm_sales_details")

## ERP Source Tables DDL (Silver Layer)
Define Silver Delta Lake tables for ERP customer locations, demographics, and product categories.

In [ ]:
# COMMAND ----------
# 4. silver.erp_loc_a101
spark.sql("DROP TABLE IF EXISTS silver.erp_loc_a101")
spark.sql("""
CREATE TABLE silver.erp_loc_a101 (
    cid STRING,
    cntry STRING,
    dwh_create_date TIMESTAMP
)
USING DELTA
COMMENT 'Standardized ERP customer geographic location mapping'
""")
print(">> Table created: silver.erp_loc_a101")

# 5. silver.erp_cust_az12
spark.sql("DROP TABLE IF EXISTS silver.erp_cust_az12")
spark.sql("""
CREATE TABLE silver.erp_cust_az12 (
    cid STRING,
    bdate DATE,
    gen STRING,
    dwh_create_date TIMESTAMP
)
USING DELTA
COMMENT 'Cleaned ERP customer demographic information with standardized gender and validated birthdates'
""")
print(">> Table created: silver.erp_cust_az12")

# 6. silver.erp_px_cat_g1v2
spark.sql("DROP TABLE IF EXISTS silver.erp_px_cat_g1v2")
spark.sql("""
CREATE TABLE silver.erp_px_cat_g1v2 (
    id STRING,
    cat STRING,
    subcat STRING,
    maintenance STRING,
    dwh_create_date TIMESTAMP
)
USING DELTA
COMMENT 'Standardized ERP product category hierarchy and maintenance classification'
""")
print(">> Table created: silver.erp_px_cat_g1v2")

## Table Verification
Confirm all 6 Silver Delta tables exist in the metastore and display their schema metadata.

In [ ]:
# COMMAND ----------
# List all tables in the silver database
df_silver_tables = spark.sql("SHOW TABLES IN silver")
display(df_silver_tables)

# Inspect details of each silver table
for table_row in df_silver_tables.collect():
    t_name = f"silver.{table_row['tableName']}"
    print(f"\n--- Schema Definition: {t_name} ---")
    spark.sql(f"DESCRIBE TABLE {t_name}").show(truncate=False)